# SMS spam detection with four NLP models
DS483 · Tasks 1 to 5



# SMS message preparation and similarity
DS483 · Student 1 · Tasks 1 and 2


In [1]:
from pathlib import Path
import hashlib
import html
import re
import unicodedata
from zipfile import ZipFile

import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedGroupKFold

with ZipFile(Path("uci_sms.zip")) as archive:
    raw = archive.read("SMSSpamCollection")
expected_hash = "7d039a24a6083ed9ef0f806ebad56bbb976e3aeb8de05669173bfdc4996c239d"
assert hashlib.sha256(raw).hexdigest() == expected_hash, "Unexpected source file"
records = []
for line_number, line in enumerate(raw.decode("utf-8").split("\n"), start=1):
    if line.strip():
        label, text = line.rstrip("\r").split("\t", 1)
        records.append((f"sms_{line_number:05d}", label, text))
sms = pd.DataFrame(records, columns=["message_id", "label", "text"])
assert sms.message_id.is_unique and set(sms.label) == {"ham", "spam"}
assert sms.text.str.strip().ne("").all()
counts = sms.label.value_counts()
print(pd.DataFrame({"messages": counts, "percent": (100 * counts / len(sms)).round(2)}).to_string())
print("Records:", len(sms), "| pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)

       messages  percent
label                   
ham        4827     86.6
spam        747     13.4
Records: 5574 | pandas: 2.2.3 | scikit-learn: 1.8.0


## Task 1 · Prepare the text



In [2]:
def clean_sms(text):
    text = unicodedata.normalize("NFKC", html.unescape(text)).casefold()
    text = text.replace("’", "'").replace("‘", "'")
    text = re.sub(r"(?:https?://|www\.)\S+", " urltoken ", text)
    text = re.sub(r"[£$€]", " moneytoken ", text)
    text = re.sub(r"\d+", " numtoken ", text)
    words = re.findall(r"[^\W\d_]+(?:'[^\W\d_]+)*", text)
    return " ".join(words)

sms["clean_text"] = sms.text.map(clean_sms)
print("Extra copies of identical raw messages:", sms.text.duplicated().sum())
print("Extra copies after cleaning:", sms.clean_text.duplicated().sum())
print("Rows empty after cleaning:", sms.clean_text.eq("").sum())
print("Clean-text groups containing both labels:", (sms.groupby("clean_text").label.nunique() > 1).sum())

Extra copies of identical raw messages: 403
Extra copies after cleaning: 477
Rows empty after cleaning: 2
Clean-text groups containing both labels: 0


## Freeze the split for all four classifiers



In [3]:
folds = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=483)
sms["split"] = "train"
for fold, (_, indices) in enumerate(folds.split(sms, sms.label, groups=sms.clean_text)):
    if fold < 2:
        sms.loc[indices, "split"] = ["test", "validation"][fold]

assert sms.groupby("clean_text").split.nunique().max() == 1
assert set(sms.split) == {"train", "validation", "test"}
split_table = pd.crosstab(sms.split, sms.label).reindex(["train", "validation", "test"])
split_table["total"] = split_table.sum(axis=1)
split_table["spam_pct"] = (100 * split_table.spam / split_table.total).round(2)
print(split_table.to_string())
sms[["message_id", "split", "label", "text", "clean_text"]].to_csv("splits.csv", index=False)
train = sms.loc[sms.split.eq("train")].reset_index(drop=True)

label        ham  spam  total  spam_pct
split                                  
train       3447   533   3980     13.39
validation   690   107    797     13.43
test         690   107    797     13.43


## Which features distinguish the training messages?



In [4]:
signals = train.assign(
    characters=train.text.str.len(),
    has_digit=train.text.str.contains(r"\d"),
    has_exclamation=train.text.str.contains("!", regex=False),
    has_currency=train.text.str.contains(r"[£$€]"),
)
profile = signals.groupby("label").agg(
    median_chars=("characters", "median"),
    digit_pct=("has_digit", "mean"),
    exclamation_pct=("has_exclamation", "mean"),
    currency_pct=("has_currency", "mean"),
)
profile.iloc[:, 1:] = profile.iloc[:, 1:] * 100
print(profile.round(2).to_string())

       median_chars  digit_pct  exclamation_pct  currency_pct
label                                                        
ham            53.0      15.61            11.52          0.38
spam          148.0      93.81            50.28         34.33


## What changes in individual messages?

Select short training examples with identifiable cues. The original and cleaned text are printed side by side by ID.

In [5]:
samples = [
    ("ham", train.text.str.contains("!", regex=False)),
    ("spam", train.text.str.contains("£", regex=False)),
]
for label, cue in samples:
    row = train.loc[train.label.eq(label) & cue & train.text.str.len().between(20, 160)].iloc[0]
    print(row.message_id, "|", label)
    print("Original:", row.text)
    print("Cleaned: ", row.clean_text, "\n")

sms_00015 | ham
Original: I HAVE A DATE ON SUNDAY WITH WILL!!
Cleaned:  i have a date on sunday with will 

sms_00013 | spam
Original: URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010 T&C www.dbuk.net LCCLTD POBOX 4403LDNW1A7RW18
Cleaned:  urgent you have won a numtoken week free membership in our moneytoken numtoken numtoken prize jackpot txt the word claim to no numtoken t c urltoken lccltd pobox numtoken ldnw numtoken a numtoken rw numtoken 



### What the training messages show



## Task 2 · Retrieve similar messages



In [6]:
vectorizer = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, norm="l2")
vectors = vectorizer.fit_transform(train.clean_text)
terms = vectorizer.get_feature_names_out()
print("Training messages:", vectors.shape[0], "| terms:", vectors.shape[1])

Training messages: 3980 | terms: 6472


In [7]:
def describe_neighbor(query_index):
    scores = cosine_similarity(vectors[query_index], vectors).ravel()
    eligible = train.clean_text.ne(train.at[query_index, "clean_text"]).to_numpy()
    scores[~eligible] = -1
    neighbor = int(np.argmax(scores))
    contributions = vectors[query_index].multiply(vectors[neighbor]).tocoo()
    strongest = np.argsort(-contributions.data)[:5]
    shared = [terms[contributions.col[j]] for j in strongest]
    print("Query:", train.at[query_index, "message_id"], "|", train.at[query_index, "label"],
          "|", train.at[query_index, "text"])
    print("Match:", train.at[neighbor, "message_id"], "|", train.at[neighbor, "label"],
          "|", train.at[neighbor, "text"])
    print(f"Cosine: {scores[neighbor]:.4f} | highest shared contributions: {', '.join(shared)}\n")
    return query_index, neighbor, float(scores[neighbor]), shared

# Choose everyday and subscription query phrases before searching for each nearest neighbor.
results = []
for label, topic in [("ham", "catch the bus"), ("spam", "ringtone"), ("ham", "sorry, i")]:
    mask = train.label.eq(label) & train.text.str.lower().str.contains(topic, regex=False)
    query_index = int(train.index[mask][0])
    results.append(describe_neighbor(query_index))

Query: sms_00028 | ham | Did you catch the bus ? Are you frying an egg ? Did you make a tea? Are you eating your mom's left over dinner ? Do you feel my Love ?
Match: sms_05020 | ham | Babe ! What are you doing ? Where are you ? Who are you talking to ? Do you think of me ? Are you being a good boy? Are you missing me? Do you love me ?
Cosine: 0.3609 | highest shared contributions: you, are, do, love, a

Query: sms_00096 | spam | Your free ringtone is waiting to be collected. Simply text the password "MIX" to 85069 to verify. Get Usher and Britney. FML, PO Box 5249, MK17 92H. 450Ppw 16
Match: sms_00402 | spam | FREE RINGTONE text FIRST to 87131 for a poly or text GET to 87131 for a true tone! Help? 0845 2814032 16 after 1st free, tones are 3x£150pw to e£nd txt stop
Cosine: 0.3626 | highest shared contributions: numtoken, to, text, ringtone, free

Query: sms_00058 | ham | Sorry, I'll call later in meeting.
Match: sms_05063 | ham | Sorry, I'll call you  later. I am in meeting sir.
Cosine

### What the retrieved pairs show



## Handoff for Tasks 3 to 5



# SMS spam classification with three models
DS483 · Student 2 · Tasks 3 and 4

Keep Layan’s `splits.csv` in the working directory. On Colab, open this notebook and upload that CSV into the Files pane. Run all cells in order. The notebook trains models on CPU and exports metrics and predictions for the BERT comparison.

Data: [UCI SMS Spam Collection](https://doi.org/10.24432/C5CC84). Derived split and preprocessing from Tasks 1 and 2. ChatGPT assisted with implementation and initial analysis.

In [8]:
from collections import Counter
from copy import deepcopy
from pathlib import Path
import hashlib
import random

import numpy as np
import pandas as pd
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

SEED = 483
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
source = Path("splits.csv")
expected = "1d59db7174d98fce90fe16541e672c207fde0e05885269cbdb47f638b2ae0712"
assert hashlib.sha256(source.read_bytes().replace(b"\r\n", b"\n")).hexdigest() == expected, "Use Layan's shared splits.csv."
data = pd.read_csv(source, keep_default_na=False)
assert data.message_id.is_unique and data.groupby("clean_text").split.nunique().max() == 1
parts = {name: data.loc[data.split.eq(name)].reset_index(drop=True) for name in ("train", "validation", "test")}
y = {name: frame.label.eq("spam").astype(int).to_numpy() for name, frame in parts.items()}
print(pd.crosstab(data.split, data.label).to_string())
print("pandas", pd.__version__, "scikit-learn", sklearn.__version__, "torch", torch.__version__)

label        ham  spam
split                 
test         690   107
train       3447   533
validation   690   107
pandas 2.2.3 scikit-learn 1.8.0 torch 2.8.0+cpu


## Task 3 · Naive Bayes and Logistic Regression



In [9]:
def choose_cutoff(labels, probabilities):
    candidates = [(f1_score(labels, probabilities >= t, zero_division=0), t)
                  for t in (0.50, 0.35, 0.65)]
    return max(candidates, key=lambda item: item[0])

tfidf = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                        ngram_range=(1, 2), sublinear_tf=True)
x_train = tfidf.fit_transform(parts["train"].clean_text)
x_val = tfidf.transform(parts["validation"].clean_text)
families = {
    "Naive Bayes": [(f"alpha={a}", MultinomialNB(alpha=a)) for a in (0.1, 0.5, 1.0)],
    "Logistic Regression": [(f"C={c}", LogisticRegression(C=c, solver="liblinear", random_state=SEED))
                            for c in (1.0, 4.0, 16.0)]
}
selected, val_rows = {}, []
for name, candidates in families.items():
    for setting, model in candidates:
        model.fit(x_train, y["train"])
        score, cutoff = choose_cutoff(y["validation"], model.predict_proba(x_val)[:, 1])
        val_rows.append((name, setting, cutoff, score))
        if name not in selected or score > selected[name]["f1"]:
            selected[name] = dict(model=model, setting=setting, cutoff=cutoff, f1=score)
print(pd.DataFrame(val_rows, columns=["model", "setting", "cutoff", "validation_spam_F1"]).round(4).to_string(index=False))

              model   setting  cutoff  validation_spam_F1
        Naive Bayes alpha=0.1    0.35              0.9665
        Naive Bayes alpha=0.5    0.35              0.8808
        Naive Bayes alpha=1.0    0.35              0.8387
Logistic Regression     C=1.0    0.35              0.9665
Logistic Regression     C=4.0    0.35              0.9668
Logistic Regression    C=16.0    0.35              0.9717


## Task 4 · Word sequences and LSTM


In [10]:
word_counts = Counter(word for text in parts["train"].clean_text for word in text.split())
vocab = {word: i for i, word in enumerate(sorted(w for w, n in word_counts.items() if n >= 2), start=2)}
lengths = parts["train"].clean_text.str.split().str.len()
max_len = max(1, min(100, int(np.ceil(np.percentile(lengths, 99)))))

def encode(messages):
    inputs = torch.zeros((len(messages), max_len), dtype=torch.long)
    for i, text in enumerate(messages):
        ids = [vocab.get(w, 1) for w in text.split()[:max_len]] or [1]
        inputs[i, :len(ids)] = torch.tensor(ids)
    return inputs

train_ids = encode(parts["train"].clean_text)
val_ids = encode(parts["validation"].clean_text)
print("Vocabulary (including special IDs):", len(vocab) + 2, "| maximum tokens:", max_len)

Vocabulary (including special IDs): 3192 | maximum tokens: 54


In [11]:
class SMSLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 32, padding_idx=0)
        self.lstm = nn.LSTM(32, 48, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.output = nn.Linear(48, 1)

    def forward(self, ids):
        hidden, _ = self.lstm(self.embedding(ids))
        hidden = hidden.masked_fill(ids.eq(0).unsqueeze(-1), float("-inf"))
        pooled = hidden.max(dim=1).values
        return self.output(self.dropout(pooled)).squeeze(1)

def predict_probabilities(model, ids):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(x).sigmoid() for x in ids.split(128)]).numpy()

In [12]:
network = SMSLSTM(len(vocab) + 2)
train_labels = torch.tensor(y["train"], dtype=torch.float32)
loader = DataLoader(TensorDataset(train_ids, train_labels), batch_size=64, shuffle=True,
                    generator=torch.Generator().manual_seed(SEED))
positive_weight = ((train_labels == 0).sum() / train_labels.sum()).sqrt()
loss_fn = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
optimizer = torch.optim.Adam(network.parameters(), lr=0.003)
best_f1, best_epoch, history = -1.0, 0, []
for epoch in range(1, 21):
    network.train()
    total_loss = 0.0
    for ids, targets in loader:
        optimizer.zero_grad()
        loss = loss_fn(network(ids), targets)
        loss.backward()
        nn.utils.clip_grad_norm_(network.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(targets)
    val_f1, threshold = choose_cutoff(y["validation"], predict_probabilities(network, val_ids))
    history.append((epoch, total_loss / len(train_labels), val_f1, threshold))
    if val_f1 > best_f1:
        best_f1, best_epoch, best_cutoff = val_f1, epoch, threshold
        best_state = deepcopy(network.state_dict())
    if epoch - best_epoch >= 4:
        break
network.load_state_dict(best_state)
print(pd.DataFrame(history, columns=["epoch", "train_loss", "validation_spam_F1", "cutoff"]).round(4).to_string(index=False))
print("Selected epoch:", best_epoch, "| cutoff:", best_cutoff, "| trainable parameters:", sum(p.numel() for p in network.parameters()))

 epoch  train_loss  validation_spam_F1  cutoff
     1      0.5055              0.8485    0.35
     2      0.1972              0.8634    0.65
     3      0.1244              0.9273    0.35
     4      0.0853              0.9099    0.50
     5      0.0603              0.9340    0.50
     6      0.0490              0.9067    0.50
     7      0.0291              0.9050    0.35
     8      0.0199              0.9224    0.50
     9      0.0170              0.9124    0.50
Selected epoch: 5 | cutoff: 0.5 | trainable parameters: 117937


### Does token order affect the fitted LSTM?



In [13]:
shuffled = val_ids.clone()
rng = np.random.default_rng(SEED)
for row in shuffled:
    n = int(row.ne(0).sum())
    row[:n] = row[:n][rng.permutation(n)].clone()
ordered_pred = predict_probabilities(network, val_ids) >= best_cutoff
shuffled_pred = predict_probabilities(network, shuffled) >= best_cutoff
print("Validation spam F1 (original, shuffled):",
      round(f1_score(y["validation"], ordered_pred), 4), round(f1_score(y["validation"], shuffled_pred), 4))
print("Changed validation predictions:", int((ordered_pred != shuffled_pred).sum()))

Validation spam F1 (original, shuffled): 0.934 0.9163
Changed validation predictions: 9


## Shared test evaluation



In [14]:
test = parts["test"]
x_test = tfidf.transform(test.clean_text)
estimates = {name: (entry["model"].predict_proba(x_test)[:, 1], entry["cutoff"], entry["setting"])
             for name, entry in selected.items()}
estimates["LSTM"] = (predict_probabilities(network, encode(test.clean_text)), best_cutoff, f"epoch={best_epoch}")
results, prediction_rows = [], []
for name, (prob, cutoff, setting) in estimates.items():
    pred = (prob >= cutoff).astype(int)
    tn, fp, fn, tp = confusion_matrix(y["test"], pred, labels=[0, 1]).ravel()
    results.append(dict(model=name, setting=setting, cutoff=cutoff, test_size=len(test),
                        accuracy=accuracy_score(y["test"], pred), spam_precision=precision_score(y["test"], pred),
                        spam_recall=recall_score(y["test"], pred), spam_f1=f1_score(y["test"], pred),
                        macro_f1=f1_score(y["test"], pred, average="macro"), TN=tn, FP=fp, FN=fn, TP=tp))
    prediction_rows.append(pd.DataFrame(dict(message_id=test.message_id, actual=test.label,
        model=name, predicted=np.where(pred, "spam", "ham"), spam_score=prob)))
metrics = pd.DataFrame(results)
predictions = pd.concat(prediction_rows, ignore_index=True)
metrics.to_csv("student2_metrics.csv", index=False)
predictions.to_csv("student2_predictions.csv", index=False)
print(metrics.round(4).to_string(index=False))

              model   setting  cutoff  test_size  accuracy  spam_precision  spam_recall  spam_f1  macro_f1  TN  FP  FN  TP
        Naive Bayes alpha=0.1    0.35        797    0.9875          0.9619       0.9439   0.9528    0.9728 686   4   6 101
Logistic Regression    C=16.0    0.35        797    0.9812          0.9107       0.9533   0.9315    0.9603 680  10   5 102
               LSTM   epoch=5    0.50        797    0.9787          0.8879       0.9626   0.9238    0.9557 677  13   4 103


## Inspect errors by model



In [15]:
mistakes = predictions[predictions.actual != predictions.predicted].merge(
    test[["message_id", "text"]], on="message_id", validate="many_to_one")
examples = mistakes.sort_values("message_id").groupby(["model", "actual"], sort=True).head(1)
for row in examples.sort_values(["model","actual"]).itertuples():
    print(f"{row.model} | {row.message_id} | {row.actual} → {row.predicted} | score {row.spam_score:.3f}")
    print(row.text, "\n")

LSTM | sms_00426 | ham → spam | score 0.716
Send this to ur friends and receive something about ur voice..... How is my speaking expression? 1.childish 2.naughty 3.Sentiment 4.rowdy 5.ful of attitude 6.romantic 7.shy 8.Attractive 9.funny  &lt;#&gt; .irritating  &lt;#&gt; .lovable. reply me.. 

LSTM | sms_00166 | spam → ham | score 0.451
BangBabes Ur order is on the way. U SHOULD receive a Service Msg 2 download UR content. If U do not, GoTo wap. bangb. tv on UR mobile internet/service menu 

Logistic Regression | sms_00426 | ham → spam | score 0.436
Send this to ur friends and receive something about ur voice..... How is my speaking expression? 1.childish 2.naughty 3.Sentiment 4.rowdy 5.ful of attitude 6.romantic 7.shy 8.Attractive 9.funny  &lt;#&gt; .irritating  &lt;#&gt; .lovable. reply me.. 

Logistic Regression | sms_02559 | spam → ham | score 0.082
This message is brought to you by GMW Ltd. and is not connected to the 

Naive Bayes | sms_01291 | ham → spam | score 0.652
Hey...Grea

## Interpreting the trade-off

Naive Bayes had the highest test spam F1 (0.9528). Logistic Regression caught one more spam message but incorrectly flagged six more legitimate ones. The LSTM caught another spam message beyond Logistic Regression and added three false alarms. The token-order shuffle changed nine validation predictions and reduced spam F1 from 0.9340 to 0.9163. Word order mattered to this fitted network; its test spam F1 still trailed both classical baselines.

## Reading the error examples

The farm-tour invitation (sms_01291) is legitimate but includes prices and a deposit, patterns also present in commercial messages. The two classical models missed the truncated company disclaimer (sms_02559). The chain-style message sms_00426 is labelled ham and was flagged by both Logistic Regression and the LSTM. The LSTM missed the order-style message sms_00166 with a spam score just below its cutoff. Textual explanations are hypotheses, not feature attributions.

## Handoff for the BERT comparison

Use the same split file and treat spam as positive. Keep training, checkpoint selection and threshold selection separate from test scoring. Join BERT predictions to student2_predictions.csv by message_id. Compare precision, recall, spam F1, FP and FN.

# Fine-tuning BERT for SMS spam detection
DS483 · Student 3 · Task 5, model development

Use Layan’s frozen split. This section trains a pretrained BERT-Tiny on raw SMS, selects an epoch and decision threshold on validation, and evaluates the selected model on the shared test messages. Keep `splits.csv` and `bert_tiny_pretrained.zip` in the notebook’s working directory. Colab users can upload both to the Files pane. See the pinned software versions in the accompanying README.



In [16]:
from copy import deepcopy
from pathlib import Path
from zipfile import ZipFile
import hashlib
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from transformers import BertForSequenceClassification, BertTokenizerFast

BERT_SEED = 483
random.seed(BERT_SEED)
np.random.seed(BERT_SEED)
torch.manual_seed(BERT_SEED)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
split_path = Path("splits.csv")
digest = hashlib.sha256(split_path.read_bytes().replace(b"\r\n", b"\n")).hexdigest()
assert digest == "1d59db7174d98fce90fe16541e672c207fde0e05885269cbdb47f638b2ae0712", "Expected Layan's frozen split"
bert_data = pd.read_csv(split_path, keep_default_na=False)
bert_parts = {name: bert_data.loc[bert_data.split.eq(name)].reset_index(drop=True)
              for name in ("train", "validation", "test")}
print(pd.crosstab(bert_data.split, bert_data.label).to_string())

label        ham  spam
split                 
test         690   107
train       3447   533
validation   690   107


In [17]:
initial_dir = Path("bert_tiny_pretrained")
initial_dir.mkdir(exist_ok=True)
with ZipFile("bert_tiny_pretrained.zip") as source_weights:
    for filename in ("config.json", "vocab.txt", "pytorch_model.bin"):
        (initial_dir / filename).write_bytes(source_weights.read(filename))
bert_tokenizer = BertTokenizerFast.from_pretrained(initial_dir, do_lower_case=True)
bert, load_info = BertForSequenceClassification.from_pretrained(
    initial_dir, num_labels=2, output_loading_info=True, attn_implementation="eager")
assert not [k for k in load_info["missing_keys"] if not k.startswith("classifier.")]
bert.config.id2label = {0: "ham", 1: "spam"}
bert.config.label2id = {"ham": 0, "spam": 1}
print("Newly initialized:", load_info["missing_keys"])
print("Transformer layers:", bert.config.num_hidden_layers, "| hidden size:", bert.config.hidden_size)
print("Trainable parameters:", sum(p.numel() for p in bert.parameters() if p.requires_grad))

Newly initialized: ['classifier.bias', 'classifier.weight']
Transformer layers: 2 | hidden size: 128
Trainable parameters: 4386178


## Training choices



In [18]:
def bert_dataset(frame):
    tokens = bert_tokenizer(frame.text.tolist(), padding="max_length", truncation=True,
                            max_length=64, return_tensors="pt")
    labels = torch.tensor(frame.label.eq("spam").astype(int).values, dtype=torch.long)
    return TensorDataset(tokens.input_ids, tokens.attention_mask, labels)

def bert_scores(model, dataset):
    model.eval()
    scores = []
    with torch.inference_mode():
        for ids, mask, _ in DataLoader(dataset, batch_size=64):
            scores.extend(model(input_ids=ids, attention_mask=mask).logits.softmax(dim=1)[:, 1].tolist())
    return np.asarray(scores)

bert_train = bert_dataset(bert_parts["train"])
bert_val = bert_dataset(bert_parts["validation"])
bert_loader = DataLoader(bert_train, batch_size=48, shuffle=True,
                         generator=torch.Generator().manual_seed(BERT_SEED))
bert_opt = torch.optim.AdamW(bert.parameters(), lr=8e-5, weight_decay=0.01)

In [19]:
bert_best_f1 = -1.0
bert_history = []
for epoch in range(1, 5):
    bert.train()
    epoch_loss = 0.0
    for ids, mask, labels in bert_loader:
        bert_opt.zero_grad()
        out = bert(input_ids=ids, attention_mask=mask, labels=labels)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(bert.parameters(), 1.0)
        bert_opt.step()
        epoch_loss += out.loss.item() * len(labels)
    scores = bert_scores(bert, bert_val)
    val_labels = bert_val.tensors[-1].numpy()
    candidates = [(float(f1_score(val_labels, scores >= t)), t) for t in (0.50, 0.35, 0.65)]
    epoch_f1, cutoff = max(candidates, key=lambda pair: pair[0])
    bert_history.append((epoch, epoch_loss / len(bert_train), cutoff, epoch_f1))
    print(f"Epoch {epoch}: loss {epoch_loss / len(bert_train):.4f}, validation F1 {epoch_f1:.4f}, cutoff {cutoff:.2f}")
    if epoch_f1 > bert_best_f1:
        bert_best_f1, bert_cutoff, bert_epoch = epoch_f1, cutoff, epoch
        bert_weights = deepcopy(bert.state_dict())
bert.load_state_dict(bert_weights)
print("Selected epoch:", bert_epoch, "| validation spam F1:", round(bert_best_f1, 4), "| cutoff:", bert_cutoff)
bert.save_pretrained("bert_sms_trained", safe_serialization=True)
bert_tokenizer.save_pretrained("bert_sms_trained")
pd.DataFrame(bert_history, columns=["epoch","train_loss","cutoff","validation_spam_f1"]).to_csv("bert_validation.csv", index=False)

Epoch 1: loss 0.2882, validation F1 0.9545, cutoff 0.35
Epoch 2: loss 0.0712, validation F1 0.9662, cutoff 0.50
Epoch 3: loss 0.0471, validation F1 0.9662, cutoff 0.50
Epoch 4: loss 0.0270, validation F1 0.9811, cutoff 0.35
Selected epoch: 4 | validation spam F1: 0.9811 | cutoff: 0.35


In [20]:
bert_test = bert_parts["test"]
bert_prob = bert_scores(bert, bert_dataset(bert_test))
bert_actual = bert_test.label.eq("spam").astype(int)
bert_pred = (bert_prob >= bert_cutoff).astype(int)
tn, fp, fn, tp = confusion_matrix(bert_actual, bert_pred, labels=[0, 1]).ravel()
bert_metrics = pd.DataFrame([dict(model="BERT-Tiny", setting=f"epoch={bert_epoch}", cutoff=bert_cutoff,
    test_size=len(bert_test), accuracy=accuracy_score(bert_actual, bert_pred),
    spam_precision=precision_score(bert_actual, bert_pred), spam_recall=recall_score(bert_actual, bert_pred),
    spam_f1=f1_score(bert_actual, bert_pred), macro_f1=f1_score(bert_actual, bert_pred, average="macro"),
    TN=tn, FP=fp, FN=fn, TP=tp)])
bert_predictions = pd.DataFrame(dict(message_id=bert_test.message_id, actual=bert_test.label,
    model="BERT-Tiny", predicted=np.where(bert_pred, "spam", "ham"), spam_score=bert_prob))
bert_metrics.to_csv("student3_metrics.csv", index=False)
bert_predictions.to_csv("student3_predictions.csv", index=False)
print(bert_metrics.round(4).to_string(index=False))
for label in ("ham","spam"):
    errors = bert_predictions[(bert_predictions.actual == label) & (bert_predictions.predicted != label)]
    if not errors.empty:
        sample = errors.iloc[0]
        print(sample.message_id, label, "predicted", sample.predicted, "|", bert_test.set_index("message_id").at[sample.message_id, "text"])

    model setting  cutoff  test_size  accuracy  spam_precision  spam_recall  spam_f1  macro_f1  TN  FP  FN  TP
BERT-Tiny epoch=4    0.35        797    0.9875           0.945       0.9626   0.9537    0.9732 684   6   4 103
sms_00411 ham predicted spam | Message:some text missing* Sender:Name Missing* *Number Missing *Sent:Date missing *Missing U a lot thats y everything is missing sent via fullonsms.com
sms_02559 spam predicted ham | This message is brought to you by GMW Ltd. and is not connected to the


## Test interpretation



## CPU inference cost


In [21]:
from time import perf_counter
from statistics import median

# Single CPU thread for all four; clock the same 64 examples in one batch.
# All tokenization/vectorization is timed from each model's prepared input text.
# Stage 1 cleaning, disk I/O and model loading are outside this measurement.
torch.set_num_threads(1)
bench_frame = bert_test.iloc[:64]
def time_batch(call):
    for _ in range(2):
        call()
    measurements = []
    for _ in range(9):
        start = perf_counter()
        call()
        measurements.append(1000 * (perf_counter() - start))
    return median(measurements)

def infer_lstm_batch():
    return predict_probabilities(network, encode(bench_frame.clean_text))

def infer_bert_batch():
    encoded = bert_tokenizer(bench_frame.text.tolist(), padding="max_length", truncation=True,
                             max_length=64, return_tensors="pt")
    bert.eval()
    with torch.inference_mode():
        return bert(**encoded).logits.softmax(1)[:,1].numpy()

latency_rows = []
for name in ("Naive Bayes", "Logistic Regression"):
    fitted = selected[name]["model"]
    ms = time_batch(lambda: fitted.predict_proba(tfidf.transform(bench_frame.clean_text)))
    latency_rows.append((name, ms))
latency_rows += [("LSTM", time_batch(infer_lstm_batch)), ("BERT-Tiny", time_batch(infer_bert_batch))]
runtime_table = pd.DataFrame(latency_rows, columns=["model", "median_ms_batch64"])
runtime_table.to_csv("cpu_timing.csv", index=False)
print(runtime_table.round(2).to_string(index=False))
print("TF-IDF features:", len(tfidf.vocabulary_), "| LSTM parameters:", sum(p.numel() for p in network.parameters()),
      "| BERT parameters:", sum(p.numel() for p in bert.parameters()))

              model  median_ms_batch64
        Naive Bayes               1.20
Logistic Regression               0.97
               LSTM               2.80
          BERT-Tiny              59.49
TF-IDF features: 38161 | LSTM parameters: 117937 | BERT parameters: 4386178


# Four-model comparison and deployment choice
DS483 · Student 4 · Task 5, comparison



In [22]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

frozen = pd.read_csv("splits.csv", keep_default_na=False).query("split == 'test'").set_index("message_id").sort_index()
assert len(frozen) == 797 and frozen.index.is_unique
pred_files = ["student2_predictions.csv", "student3_predictions.csv"]
score_files = ["student2_metrics.csv", "student3_metrics.csv"]
stacked = pd.concat([pd.read_csv(p) for p in pred_files], ignore_index=True)
reported = pd.concat([pd.read_csv(p) for p in score_files], ignore_index=True).set_index("model")
comparison_rows = []
for name, group in stacked.groupby("model", sort=False):
    group = group.set_index("message_id").sort_index()
    assert group.index.equals(frozen.index) and group.actual.equals(frozen.label)
    true = group.actual.eq("spam")
    predicted = group.predicted.eq("spam")
    assert np.array_equal(group.spam_score >= reported.at[name, "cutoff"], predicted)
    tn, fp, fn, tp = confusion_matrix(true, predicted, labels=[False, True]).ravel()
    row = dict(model=name, accuracy=accuracy_score(true,predicted), spam_precision=precision_score(true,predicted),
               spam_recall=recall_score(true,predicted), spam_f1=f1_score(true,predicted),
               macro_f1=f1_score(true,predicted,average="macro"), TN=tn, FP=fp, FN=fn, TP=tp)
    assert all(np.isclose(value, reported.at[name,key]) for key,value in row.items() if key != "model")
    comparison_rows.append(row)
summary = pd.DataFrame(comparison_rows)
assert set(summary.model) == {"Naive Bayes","Logistic Regression","LSTM","BERT-Tiny"}
summary.to_csv("four_model_metrics.csv", index=False)
print(summary.round(4).to_string(index=False))

              model  accuracy  spam_precision  spam_recall  spam_f1  macro_f1  TN  FP  FN  TP
        Naive Bayes    0.9875          0.9619       0.9439   0.9528    0.9728 686   4   6 101
Logistic Regression    0.9812          0.9107       0.9533   0.9315    0.9603 680  10   5 102
               LSTM    0.9787          0.8879       0.9626   0.9238    0.9557 677  13   4 103
          BERT-Tiny    0.9875          0.9450       0.9626   0.9537    0.9732 684   6   4 103


In [23]:
wide = stacked.pivot(index="message_id", columns="model", values="predicted").loc[frozen.index]
wide.insert(0, "actual", frozen.label)
correct = wide.drop(columns="actual").eq(frozen.label, axis=0)
wide["models_correct"] = correct.sum(axis=1)
wide.to_csv("four_model_predictions.csv")
print("All four correct:", int(wide.models_correct.eq(4).sum()))
print("All four wrong:", int(wide.models_correct.eq(0).sum()))
print("BERT correct / NB wrong:", int((correct["BERT-Tiny"] & ~correct["Naive Bayes"]).sum()))
print("NB correct / BERT wrong:", int((~correct["BERT-Tiny"] & correct["Naive Bayes"]).sum()))
for key in ["sms_01291", "sms_03825", "sms_01613"]:
    print(key, "|", frozen.at[key,"text"])
    print(wide.loc[key].to_string(), "\n")

All four correct: 766
All four wrong: 3
BERT correct / NB wrong: 5
NB correct / BERT wrong: 5
sms_01291 | Hey...Great deal...Farm tour 9am to 5pm $95/pax, $50 deposit by 16 May
model
actual                  ham
BERT-Tiny               ham
LSTM                   spam
Logistic Regression    spam
Naive Bayes            spam
models_correct            1 

sms_03825 | Please protect yourself from e-threats. SIB never asks for sensitive information like Passwords,ATM/SMS PIN thru email. Never share your password with anybody.
model
actual                  ham
BERT-Tiny              spam
LSTM                    ham
Logistic Regression     ham
Naive Bayes             ham
models_correct            3 

sms_01613 | 645
model
actual                  ham
BERT-Tiny              spam
LSTM                   spam
Logistic Regression    spam
Naive Bayes            spam
models_correct            0 



In [24]:
timing = pd.read_csv("cpu_timing.csv")
print(timing.round(2).to_string(index=False))

              model  median_ms_batch64
        Naive Bayes               1.20
Logistic Regression               0.97
               LSTM               2.80
          BERT-Tiny              59.49
